# Gauss-Newton Matrix

In [ ]:
import numpy as np
from scipy.spatial.transform import Rotation as R

MU_EARTH = 3.986004418*(10**14)
SPEED_OF_LIGHT = 299792458

## COE2RV

NOTE: Need to use semiparameter ($p$) instead of semimajor axis ($a$). $a$ is infinite for the parabola, whereas $p$ is defined for all orbits

In [ ]:
def COE2RV(coe, mu=MU_EARTH):

    # INPUT: 
    #   coe is an array of the Keplerian Orbital Elements
    #       a - semi-major axis
    #       e - eccentricity
    #       i - inclination
    #       node - right ascension of the ascending node
    #       arg - argument of perigee
    #       nu - true anomaly
    #   mu - gravitational parameters (=GM). Default set to the value for Earth

    # OUTPUT:
    #   r - position vector of satellite
    #   v - velocity vector of satellite


    a, e, i, node, arg, nu = coe

    sin_nu = np.sin(np.deg2rad(nu))
    cos_nu = np.cos(np.deg2rad(nu))

    # Calculate semiparameter (p)
    p = a * (1-e**2)

    # Perifocal Coordinate System
    r_PQW = np.zeros(3)
    v_PQW = np.zeros(3)

    r_PQW[0] = (p * cos_nu) / (1 + e*cos_nu)
    r_PQW[1] = (p * sin_nu) / (1 + e*cos_nu)
    
    v_PQW[0] = -1 * np.sqrt(mu/p) * sin_nu
    v_PQW[1] = np.sqrt(mu/p) * (e + cos_nu)

    # Rotation Matrix
    R_node_z = R.from_euler('z', np.deg2rad(node)).as_matrix()
    R_i_x    = R.from_euler('x', np.deg2rad(i)).as_matrix()
    R_arg_z  = R.from_euler('z', np.deg2rad(arg)).as_matrix()
    R_total  = R_node_z @ R_i_x @ R_arg_z

    # Rotate Perifocal to IJK
    r_IJK = R_total @ r_PQW
    v_IJK = R_total @ v_PQW

    return r_IJK, v_IJK, R_total

### Example 2.6 from FoAaA (pg. 119)

Verifying function is correct

In [46]:
p = 11067.790 #km
e = 0.83285
i = 87.87
node = 227.89
arg = 53.38
nu = 92.335

a = (p*1000) / (1-e**2)

coe = [a, e, i, node, arg, nu]

r, v, rot_mat = COE2RV(coe)

print("Vector r (m):")
print("Textbook: [6525344  6861535  6449125]")
print("Mine:    ", r)
print("Shape:   ", r.shape)

print("\nVector v (m/s):")
print("Textbook: [4902.276  5533.124  -1975.709]")
print("Mine:    ", v)
print("Shape:   ", v.shape)

Vector r (m):
Textbook: [6525344  6861535  6449125]
Mine:     [6525368.12098609 6861531.83489605 6449118.61416016]
Shape:    (3,)

Vector v (m/s):
Textbook: [4902.276  5533.124  -1975.709]
Mine:     [ 4902.27864642  5533.13956836 -1975.71009954]
Shape:    (3,)


## Calculate $\nu$ from $M_0$

- $n$ is the mean motion
$$
M(t) = M_0 + n (t-t_0)
$$
$$
n = \sqrt{\frac{\mu}{a^3}}
$$
- Use Newton-Raphson Method (Algorithm 2 in FoAaA pg. 65) to find $E$
	- Converts $M=E-e\sin (E)$ to be in terms of $E$
    - Tolerance set to $10^{-8}$ as according to textbook
- The calculate $\nu$
$$
\cos(\nu) = \frac{\cos(E) - e}{1-e\cos(E)}
$$

In [ ]:
def nCalc(a, mu=MU_EARTH):
    # Returns the mean motion in rads/s
    return np.sqrt(mu / a**3)

In [40]:
def NewtRaph(M, e, tolerance=10**-8):
    # Converts single pair of M and e to E
    # Inputs:
    #   M - Mean anomaly (degrees)
    #   e - Eccentricity
    #   tolerance - default to 10^8
    # Outputs:
    #   E - Eccentric anomaly
    
    M_rad = np.deg2rad(M)

    if (M_rad>-np.pi and M_rad<0) or (M_rad>np.pi):
        E = M_rad-e
    else:
        E = M_rad+e

    while True:
        sin_E = np.sin(E)
        cos_E = np.cos(E)
        nextE = E + (M_rad-E+e*sin_E)/(1-e*cos_E)

        abs_diff = abs(nextE-E)
        E = nextE

        if abs_diff < tolerance:
            break

    return E

In [ ]:
def MultiNewtRaph(t, M_0, n, e, tolerance=10**-8):
    # Inputs:
    #   t - array of time entries (s)
    #   M_0 - initial mean anomaly (deg)
    #   n - mean motion (rads/s) 
    #   e - eccentricity
    # Outputs:
    #   nu_t - true anomaly (in radians) at each point in time

    t_0 = t[0]

    # Mean anomalies
    M_t = np.zeros_like(t)
    M_t = M_0 + n*(t-t_0)

    # Newton-Raphson to find eccentric anomaly (E)
    E_t = np.zeros_like(M_t)
    for i in range(len(M_t)):
        E_t[i] = NewtRaph(M[i], e, tolerance=tolerance)

    return E_t

In [ ]:
def nuCalc(E, e):
    # Inputs:
    #   E - Eccentric anomaly (radians)
    #   e - Eccentricity
    # Outputs:
    #   nu - True Anomaly (radians)
     
    cosE = np.cos(E)
    nu = np.arccos( (cosE - e)/(1 - e*cosE) ) # in radians
    return nu

### Newton-Raphson: Testing Example 2-1 from FoAaA

In [42]:
M = 235.4
e = 0.4
tolerance = 10**-8

E = NewtRaph(M, e)

print("Expect E (deg): 220.512074767522")
print("Final E (deg): ", np.rad2deg(E))

Expect E (deg): 220.512074767522
Final E (deg):  220.51207476752208


---
## Groundstation Vectors

<font color='red'>TODO</font>

Position $\textbf{r}_{gs}$
1. Convert geodetic coordinates to (lat/lon/att) to ECEF
2. Rotate ECEF to ECI using Earth's rotation angle at each timestamp (via sidereal time)

Velocity $\textbf{v}_{gs}$
1. Differentiate rotation (angular velocity crossed with position)

---
## Doppler Shift from $\textbf{r}$ and $\textbf{v}$

In [ ]:
def rhoCalc(r, r_gs):
    return r-r_gs

def rho_hatCalc(rho):
    return rho / np.sqrt(rho.dot(rho))

def v_relCalc(v, v_gs):
    return v-v_gs

def kCalc(f_c, c=SPEED_OF_LIGHT):
    return f_c/c

def fDCalc(k, rho_hat, v_rel):
    return k * rho_hat * (-v_rel)

## Find Doppler Shift Error

In [ ]:
def residual(y_pred, y_true):
    return y_pred-y_true

---

## Gradient Calculations

In [ ]:
def dfD_dXCalc(k, v_rel, rho, rho_hat):
    # Relationship between Doppler shift and satellite's cartesian state

    mag_rho = np.sqrt(rho.dot(rho))

    I3 = np.eye(3)

    dfD_dr = -k * v_rel.T @ ( I3/mag_rho - (np.outer(rho, rho))/(mag_rho**3) )
    dfD_dv = -k * rho_hat.T
    return np.concatenate([dfD_dr, dfD_dv])


def dX_dtCalc(v, r, mu=MU_EARTH):
    # Derivative of the satellite's cartesian state

    mag_r = np.sqrt(r.dot(r))
    dv_dt = -mu * r / (mag_r**3)

    return np.concatenate([v, dv_dt])

In [ ]:
def dfD_dM_0Calc(dfD_dX, dX_dt, n):
    # Initial Mean Anomaly Gradient
    # Inputs:
    #   dfD_dX  - Change in Doppler measurement w.r.t. satellite's cartesian state (1x6 array)
    #   dX_dt   - Change in satellite's cartesian state w.r.t. time (6x1 array)
    #   n      - Mean motion (rad/s)
    # Outputs:
    #   dfD_dM_0 - Change in Doppler shift w.r.t. Initial Mean Anomaly (Hz/deg)

    dM_dt_reciprocal = 1 / np.rad2deg(n)

    return dfD_dX @ dX_dt * dM_dt_reciprocal

In [ ]:
def dfD_daCalc(dfD_dX, dX_dt, t, rot_mat, n, a, e, E, nu, mu=MU_EARTH):
    # Semi-Major Axis Gradient
    # Inputs:
    #   dfD_dX  - Change in Doppler measurement w.r.t. satellite's cartesian state (1x6 array)
    #   dX_dt   - Change in satellite's cartesian state w.r.t. time (6x1 array)
    #   t       - Time of observation (s)
    #   rot_mat - Perifocal-to-ECI rotation matrix (from COE2RV)
    #   n       - Mean motion (rad/s)
    #   a       - Semi-major axis (m)
    #   e       - Eccentricity
    #   E       - Eccentric anomaly (rad)
    #   nu      - True anomaly (rad)
    #   mu      - Gravitational parameter (m^3/s^2) (Defaults to Earth's mu value)
    # Outputs:
    #   dfD_da  - Change in Doppler shift w.r.t semi-major axis
    # NOTE: For calculations in this function, M is in radians

    nu_arr = np.array([np.cos(nu), np.sin(nu), 0])
    E_arr = np.array([-np.sin(E), np.sqrt(1-e**2) * np.cos(E), 0])

    dr_da_M = (1 - e*np.cos(E)) * rot_mat @ nu_arr
    dv_da_M = -n / (2- 2*e*np.cos(E)) * rot_mat @ E_arr

    dX_da_M = np.concatenate([dr_da_M, dv_da_M])
    dX_dM = dX_dt / n
    dM_da = -3/2 * np.sqrt(mu / (a**5)) * t

    dX_da = dX_da_M + dX_dM * dM_da

    return dfD_dX @ dX_da    

---
## Jacobian Construction

In [ ]:
def Jacobian(dfD_dM_0, dfD_da):
    # Create Jacobian Matrix
    # Inputs:
    #   dfD_dM_0 - Initial mean anomaly gradient (scalar or Nx1 array)
    #   dfD_da   - Semi-major axis gradient (scalar or Nx1 array)
    # Outputs:
    #   J - Outputs Nx2 array, where N is the number of observations 

    # Convert scalar values to 1D arrays
    dfD_dM_0 = np.atleast_1d(dfD_dM_0)
    dfD_da = np.atleast_1d(dfD_da)

    N = len(dfD_dM_0)       # Number of entries

    # Create matrix
    J = np.zeros((N, 2))
    for i in range(N):
        J[i] = np.array([dfD_dM_0[i], dfD_da[i]])

    return J

---

## Gauss-Newton Matrix

In [ ]:
def GaussNewton(J):
    # Create Gauss-Newton Matrix
    # Inputs:
    #   J - Nx2 Jacobian Matrix, where N is the number of observations
    # Outputs:
    #   G - 2x2 Gauss-Newton Matrix
     
    N = J.shape[0]
    return 1/N * J.T @ J

---
# Workflow for Single Observation

In [ ]:
# TODO: Make it work with given data from Kyle

# Get Orbital Parameters
# TODO: Confirm some test values
a = 1
e = 1
i = 1
node = 1
arg = 1
M_0 = 1

fD_true = -9725.11702260135
fc = 4.38 * 10**8  # Central freq
t = 0

E = NewtRaph(M_0, e)
nu = nuCalc(E, e)
n = nCalc(a)

coe = [a, e, i, node, arg, nu]

# Convert COE to State Vector (r, v)
r, v, rot_mat = COE2RV(coe)

# Groundstation - lat/lon/att to ECI 
# TODO: Finish this
r_gs = ...
v_gs = ...

# Predict Doppler Shift
k = kCalc(fc)
rho = rhoCalc(r, r_gs)
rho_hat = rho_hatCalc(rho)
v_rel = v_relCalc(v, v_gs)
fD_pred = fDCalc(k, rho_hat, v_rel)

delta_fD = residual(fD_pred, fD_true)

# Chain Rule
dfD_dX = dfD_dXCalc(k, v_rel, rho, rho_hat)
dX_dt = dX_dtCalc(v, r)

dfD_dM_0 = dfD_dM_0Calc(dfD_dX, dX_dt, n)
dfD_da = dfD_daCalc(dfD_dX, dX_dt, t, rot_mat, n, a, e, E, nu)

# Jacobian Row
J = Jacobian(dfD_dM_0, dfD_da)

# Gauss-Newton Matrix
G = GaussNewton(J)

# Eigenvalues and Eigenvectors
eig_w, eig_v = np.linalg.eig(G)